# Simulation control

Supplementary figure explaining and validating the "simulated data" control used to
compare RS/OF (running-speed / optic-flow) integration between free locomotion
(closed loop, spheres) and the motorised-wheel (treadmill) condition.

For each neuron, a simulated response is built by (1) taking the RS/OF Gaussian fit
obtained on the treadmill data and forcing it to be circular (isotropic in log(RS) and
log(OF)), then (2) predicting a dF/F trace from that circularised fit given the actual
RS/OF trajectory of a recording, and (3) convolving the predicted trace with a
biexponential calcium kernel. If the real (non-simulated) fit ellipses are more
elongated/oriented than this simulated-circular control, it indicates genuine RS/OF
integration rather than an artefact of the calcium indicator's dynamics.

This figure uses only the revision ("colasa_3d-vision_revisions") treadmill sessions
(see `v1_depth_map/revisions/revision_sessions.py`, sessions tagged `"motor"`).

See `v1_depth_map/revisions/treadmill.ipynb` and
`v1_depth_map/presentations/treadmill_20260501_poster_swc.ipynb` for the exploratory
analysis this figure is drawn from.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib

In [ ]:
import matplotlib.pyplot as plt
import flexiznam as flz
from cottage_analysis.analysis import fit_gaussian_blob as fit_gb
from cottage_analysis.analysis.spheres.simulation import make_biexponential_kernel
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting.rsof_plots import plot_RS_OF_fit, plot_RS_OF_matrix
from cottage_analysis.plotting import style

from v1_depth_map.paths import get_figures_roots
from v1_depth_map.figure_utils import treadmill
from v1_depth_map.figure_utils.rsof_integration import (
    plot_angle_eccentricity_polar,
)

cm = 1 / 2.54

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

## Load data

Revision (colasa) project only.


In [ ]:
project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Population data (all colasa treadmill/"motor" sessions), including the
# simulated-response dataframes used for the polar-scatter panels below.
TDECAY = 2
TRISE = 0.15

(
    neurons_df,
    simul_df_treadmill,
    simul_df_spheres,
    valid_sessions,
    treadmill_sessions,
) = treadmill.load_treadmill_population_neurons_df(
    flexilims_session,
    protocol_base="SpheresTubeMotor",
    tdecay=TDECAY,
    trise=TRISE,
    load_simulated=True,
    # Also load the simulated fits run with the real trial-average plateau configuration
    # (precompute_data/fit_revision_simulation.py). Merged into `simul_df_treadmill` with
    # the `treadmill.TA_SIM_SUFFIX` suffix, alongside their ellipse geometry.
    load_simulated_trial_average=True,
)

In [ ]:
# Example session/cell used for the schematic panels (kernel, circularising fit,
# simulated trace). Same example as in the exploratory notebooks.
EXAMPLE_SESSION = "PZAG17.3a_S20250402"
EXAMPLE_CELL = "PZAG17.3a_S20250402_83"
example_mouse, example_session = EXAMPLE_SESSION.split("_")

ndf, trials_df_tm, trials_df_sphere = pipeline_utils.load_treadmill_and_sphere_datasets(
    project,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
)

suite2p_ds = flz.get_datasets(
    origin_name=EXAMPLE_SESSION,
    dataset_type="suite2p_rois",
    filter_datasets={"annotated": True},
    flexilims_session=flexilims_session,
    allow_multiple=False,
)
fs = suite2p_ds.extra_attributes["fs"]

example_cell = neurons_df[neurons_df.roi_uid == EXAMPLE_CELL].iloc[0]
roi = example_cell.roi

## Build the circularised-fit control for the example cell

Force the treadmill RS/OF Gaussian fit to be circular (isotropic), swap in the
simulated dF/F for that ROI, and compare the resulting fit to the real one.


In [ ]:
rs_bins, of_bins, tick_dict = treadmill.compute_treadmill_rsof_bins(trials_df_tm)
range_kwargs = dict(
    log_range={"log_base": 2}, rs_bins=rs_bins, of_bins=of_bins, tick_dict=tick_dict
)

# Simulated dF/F trace for this ROI (treadmill condition).
trace_ds = flz.get_datasets(
    origin_name=EXAMPLE_SESSION,
    dataset_type="neurons_df",
    flexilims_session=flexilims_session,
    allow_multiple=False,
)
trace_path = trace_ds.path_full.with_name(
    "simulated_traces_treadmill_trial_average_plateau"
    f"_{TDECAY}_{TRISE}_circular.parquet"
)
simul_traces = pd.read_parquet(trace_path)
simul_data = simul_traces[simul_traces.roi == example_cell.roi]
assert len(simul_data) == 1
simul_data = simul_data.iloc[0]

trials_df_tm_simul = trials_df_tm.copy()
trials_df_tm_simul["dff_stim"] = [dff.copy() for dff in trials_df_tm.dff_stim]
trials_df_tm_simul.dff_stim += np.nan
indices = np.hstack([0, trials_df_tm.dff_stim.map(len).values.cumsum()])
assert indices[-1] == len(simul_data.fake_dff), (
    f"trials_df_tm has {indices[-1]} total frames but fake_dff has "
    f"{len(simul_data.fake_dff)} -- trial boundaries no longer match the trace file. "
    "Both should be plateau-cut; regenerate with "
    "`fit_revision_simulation.py --traces --no-fits --redo` if the onset detection has "
    "changed."
)
for i in range(len(trials_df_tm_simul["dff_stim"])):
    dff_trial = trials_df_tm_simul.at[i, "dff_stim"]
    dff_trial[:, roi] = simul_data.fake_dff[indices[i] : indices[i + 1]]
    trials_df_tm_simul.at[i, "dff_stim"] = dff_trial

# Circularised copy of every ROI's fit (needed by plot_RS_OF_fit's sfx="_circular_sim").
# Seeded from the trial-average fits, i.e. the same popts
# `fit_revision_simulation.py` circularised to produce the trace spliced in above, so this
# panel shows the ground truth that actually generated the simulated matrix beside it.
popt_list = []
ndf = ndf.copy()
for popt in ndf["rsof_popt_closedloop_g2d_treadmill_trial_average_plateau"].values:
    if popt is None or np.isnan(popt).any():
        popt_model = None
    else:
        # Reduce the major axis to match the minor axis -> isotropic/circular fit
        popt_model = popt.copy()
        popt_model[3] = popt_model[4] = min(popt[3:5])
    popt_list.append(popt_model)
ndf["rsof_popt_closedloop_g2d_circular_sim"] = popt_list
ndf["rsof_test_rsq_closedloop_g2d_circular_sim"] = ndf[
    "rsof_test_rsq_closedloop_g2d_treadmill_trial_average_plateau"
]

## Build the same circularised-fit control for free locomotion (spheres)

Same as above, but using the closed-loop (spheres) fit and RS/OF trajectory instead
of the treadmill ones.


In [ ]:
# Simulated dF/F trace for this ROI (free-locomotion/sphere condition)
simul_data_sphere = simul_df_spheres[simul_df_spheres.roi_uid == example_cell.roi_uid]
assert len(simul_data_sphere) == 1
simul_data_sphere = simul_data_sphere.iloc[0]

# Unlike treadmill trial cropping, sphere trial cropping has no tunable
# onset-detection "method" (it is cut deterministically from photodiode-derived
# stim transitions), and `simulate_and_fit_session` reuses the same sync/trial-
# extraction defaults as `trials_df_sphere` above -- so no special reload is
# needed here to recover matching per-trial frame counts.
trials_df_sphere_simul = trials_df_sphere.copy()
trials_df_sphere_simul["dff_stim"] = [dff.copy() for dff in trials_df_sphere.dff_stim]
trials_df_sphere_simul.dff_stim += np.nan
indices_sphere = np.hstack([0, trials_df_sphere.dff_stim.map(len).values.cumsum()])
assert indices_sphere[-1] == len(simul_data_sphere.fake_dff), (
    f"trials_df_sphere has {indices_sphere[-1]} total frames but fake_dff has "
    f"{len(simul_data_sphere.fake_dff)} -- trial boundaries no longer match the "
    "precomputed simulated-response file; re-check the sphere sync/trial-extraction "
    "kwargs used here against however `simulate_and_fit_session` generated it."
)
for i in range(len(trials_df_sphere_simul["dff_stim"])):
    dff_trial = trials_df_sphere_simul.at[i, "dff_stim"]
    dff_trial[:, roi] = simul_data_sphere.fake_dff[
        indices_sphere[i] : indices_sphere[i + 1]
    ]
    trials_df_sphere_simul.at[i, "dff_stim"] = dff_trial

# Circularised copy of every ROI's free-locomotion (closed-loop, spheres) fit
# (needed by plot_RS_OF_fit's sfx="_circular_sim_free")
popt_list_free = []
for popt in ndf["rsof_popt_closedloop_g2d"].values:
    if popt is None or np.isnan(popt).any():
        popt_model = None
    else:
        # Reduce the major axis to match the minor axis -> isotropic/circular fit
        popt_model = popt.copy()
        popt_model[3] = popt_model[4] = min(popt[3:5])
    popt_list_free.append(popt_model)
ndf["rsof_popt_closedloop_g2d_circular_sim_free"] = popt_list_free
ndf["rsof_test_rsq_closedloop_g2d_circular_sim_free"] = ndf[
    "rsof_test_rsq_closedloop_g2d"
]

## Prepare the example sphere trial trace

Running speed / optic flow for a couple of example sphere trials, and the dF/F
response predicted by the example cell's (real) circularised Gaussian fit, convolved
with the biexponential calcium kernel.


In [ ]:
# Sphere trials used for the simulated-trace schematic
example_sphere_indices = np.arange(29, 31)

data_sphere = np.array([])
of_sphere = np.array([])
stim_part_sphere = np.array([])

for itrial, idx in enumerate(example_sphere_indices):
    trial_series = trials_df_sphere.iloc[idx]

    if itrial == 0:
        # Start with an initial blank period
        data_sphere = trial_series.RS_blank_pre[-20:]
        of_sphere = np.zeros_like(data_sphere)
        stim_part_sphere = np.zeros(data_sphere.shape, dtype=int)

    data_sphere = np.hstack([data_sphere, trial_series.RS_stim, trial_series.RS_blank])
    of_sphere = np.hstack(
        [of_sphere, trial_series.OF_stim, np.zeros_like(trial_series.RS_blank)]
    )

    stim_id = trial_series.name
    stim_part_sphere = np.hstack(
        [
            stim_part_sphere,
            np.ones(trial_series.RS_stim.shape) * stim_id,
            np.zeros(trial_series.RS_blank.shape),
        ]
    )

time_axis_sphere = np.arange(len(data_sphere)) / fs

popt = example_cell.rsof_popt_closedloop_g2d_treadmill
with np.errstate(divide="ignore", invalid="ignore"):
    rs_log = np.log(data_sphere)
    of_log = np.log(np.degrees(of_sphere))
# Handle log(0) frames (blank periods) - map to a very negative value
rs_log[np.isnan(rs_log) | np.isinf(rs_log)] = -10
of_log[np.isnan(of_log) | np.isinf(of_log)] = -10

pred_dff_sphere = fit_gb.gaussian_2d((rs_log, of_log), *popt, min_sigma=0.25)
# Same kernel builder as the actual simulation pipeline (previously duplicated
# locally as `treadmill.make_simulation_kernel`); peak-normalized ("max") to
# match its new default. `make_biexponential_kernel` only returns the kernel
# array (no time vector), so rebuild the matching time axis for the schematic.
kernel_norm = make_biexponential_kernel(
    tau_decay=TDECAY, tau_rise=TRISE, frame_rate=fs, normalization="max"
)
time_kernel = np.arange(len(kernel_norm)) / fs
sim_dff_sphere = np.convolve(pred_dff_sphere, kernel_norm, mode="full")[
    : len(pred_dff_sphere)
]

## Assemble the figure


In [ ]:
import matplotlib.patches as patches

cm = 1 / 2.54

# Two rows: on top, A the simulation schematic (left) beside B the example RS/OF matrices
# (right); below, C the two population polar plots, one under each. Everything is placed in
# CENTIMETRES from the bottom-left corner via `ax_cm`/`pt_cm`, so the figure size can change
# without distorting a panel - unlike raw figure fractions, where every height silently
# rescales with the figure.
FIG_W_CM, FIG_H_CM = 18.0, 15.5
fig = plt.figure(figsize=(FIG_W_CM * cm, FIG_H_CM * cm))
fontsize_dict = FONTSIZE_DICT


def ax_cm(x, y, w, h):
    """Axes rect [left, bottom, width, height] from cm off the bottom-left corner."""
    return [x / FIG_W_CM, y / FIG_H_CM, w / FIG_W_CM, h / FIG_H_CM]


def pt_cm(x, y):
    """A point in `ax_bg`/figure coordinates (0-1 over the figure) from cm."""
    return x / FIG_W_CM, y / FIG_H_CM


# Background axes for whole-figure coordinates, text, and arrows
ax_bg = fig.add_axes([0, 0, 1, 1])
ax_bg.set_xlim(0, 1)
ax_bg.set_ylim(0, 1)
ax_bg.axis("off")


CBAR_GAP_CM = 0.22  # last matrix -> colorbar
CBAR_W_CM = 0.09


def add_matrix_colorbar(fig, ax, vmin, vmax, fontsize_dict):
    pos = ax.get_position()
    cbar_h = pos.height * 0.5
    cax = fig.add_axes(
        [
            pos.x1 + CBAR_GAP_CM / FIG_W_CM,
            pos.y0 + (pos.height - cbar_h) / 2,
            CBAR_W_CM / FIG_W_CM,
            cbar_h,
        ]
    )
    cbar = fig.colorbar(ax.images[0], cax=cax)
    cbar.set_ticks([vmin, vmax])
    cax.tick_params(labelsize=fontsize_dict.get("legend", 10), length=2, pad=2)
    return cax


# =========================================================================
# ROW 1, LEFT (A): FITTING PROCEDURE SCHEMATIC
# =========================================================================

# A. Circularise fit boxes
ax_bg.text(
    *pt_cm(1.58, 15.06),
    "Circularise fit",
    fontsize=fontsize_dict["label"],
    ha="center",
    va="bottom",
)

ax_box_real = fig.add_axes(ax_cm(0.63, 13.20, 0.78, 1.65))
plot_RS_OF_fit(
    neurons_df=ndf,
    roi=roi,
    model="g2d",
    sfx="",
    ax=ax_box_real,
    cbar_width=None,
    label_r2=False,
    vmin=0,
    vmax=0.8,
    fontsize_dict=fontsize_dict,
    xlabel="Running speed",
    ylabel="Optic flow",
    **range_kwargs,
)
ax_box_real.set_xticks([])
ax_box_real.set_yticks([])
ax_box_real.set_title("")

# Arrow between fit and circularised fit
ax_bg.annotate(
    "",
    xy=pt_cm(1.76, 14.01),
    xytext=pt_cm(1.44, 14.01),
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=1.2,
        headwidth=4.5,
        headlength=4.5,
    ),
)

ax_box_circ = fig.add_axes(ax_cm(1.80, 13.20, 0.78, 1.65))
plot_RS_OF_fit(
    neurons_df=ndf,
    roi=roi,
    model="g2d",
    sfx="_circular_sim_free",
    ax=ax_box_circ,
    cbar_width=None,
    label_r2=False,
    vmin=0,
    vmax=0.8,
    fontsize_dict=fontsize_dict,
    xlabel="",
    ylabel="",
    **range_kwargs,
)
ax_box_circ.set_xticks([])
ax_box_circ.set_yticks([])
ax_box_circ.set_title("")

# Arrow from Circularised fit down-right to operator
ax_bg.annotate(
    "",
    xy=pt_cm(3.56, 11.91),
    xytext=pt_cm(2.66, 13.20),
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=1.2,
        headwidth=4.5,
        headlength=4.5,
    ),
)

# B. Input traces (Bottom-Left)
ax_bg.text(
    *pt_cm(1.58, 12.26),
    "Running speed",
    fontsize=fontsize_dict["tick"],
    ha="center",
    va="bottom",
)
ax_rs = fig.add_axes(ax_cm(0.45, 11.20, 2.16, 0.94))
ax_rs.plot(time_axis_sphere, data_sphere * 100, color="black", linewidth=1.5)
for trial_id in np.unique(stim_part_sphere[stim_part_sphere > 0]):
    mask = stim_part_sphere == trial_id
    ax_rs.axvspan(
        time_axis_sphere[mask][0],
        time_axis_sphere[mask][-1],
        color="gray",
        alpha=0.1,
    )
for spine in ax_rs.spines.values():
    spine.set_visible(False)
ax_rs.set_xticks([])
ax_rs.set_yticks([])

ax_bg.text(
    *pt_cm(1.58, 10.83),
    "Optic flow",
    fontsize=fontsize_dict["tick"],
    ha="center",
    va="bottom",
)
ax_of = fig.add_axes(ax_cm(0.45, 9.77, 2.16, 0.94))
ax_of.plot(time_axis_sphere, np.degrees(of_sphere), color="red", linewidth=1.5)
for trial_id in np.unique(stim_part_sphere[stim_part_sphere > 0]):
    mask = stim_part_sphere == trial_id
    ax_of.axvspan(
        time_axis_sphere[mask][0],
        time_axis_sphere[mask][-1],
        color="gray",
        alpha=0.1,
    )
for spine in ax_of.spines.values():
    spine.set_visible(False)
ax_of.set_xticks([])
ax_of.set_yticks([])

# Arrow from input traces up-right to operator
ax_bg.annotate(
    "",
    xy=pt_cm(3.56, 11.48),
    xytext=pt_cm(2.66, 10.40),
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=1.2,
        headwidth=4.5,
        headlength=4.5,
    ),
)

# C. Kernel (Top-Middle)
ax_bg.text(
    *pt_cm(3.78, 14.34),
    f"Exponential\ndecay $\\tau$ = {TDECAY}s",
    fontsize=fontsize_dict["tick"],
    ha="center",
    va="bottom",
)
ax_kernel = fig.add_axes(ax_cm(3.24, 12.98, 0.99, 1.09))
ax_kernel.plot(time_kernel, kernel_norm, color="k", lw=2)
for spine in ax_kernel.spines.values():
    spine.set_visible(False)
ax_kernel.set_xticks([])
ax_kernel.set_yticks([])

# Arrow from kernel down-left to operator
ax_bg.annotate(
    "",
    xy=pt_cm(3.74, 11.98),
    xytext=pt_cm(3.92, 12.77),
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=1.2,
        headwidth=4.5,
        headlength=4.5,
    ),
)

# D. Convolution Operator Circle with cross
op_x, op_y = pt_cm(3.78, 11.69)
# Convolution operator. `ax_bg` spans 0-1 over a non-square figure, so a `Circle` with one
# radius comes out elliptical (stretched by FIG_W_CM / FIG_H_CM); an Ellipse whose axes are
# scaled by each dimension is a true circle in cm.
OP_R_CM = 0.21
circ_op = patches.Ellipse(
    (op_x, op_y),
    width=2 * OP_R_CM / FIG_W_CM,
    height=2 * OP_R_CM / FIG_H_CM,
    edgecolor="k",
    facecolor="white",
    lw=1.5,
)
ax_bg.add_patch(circ_op)
# An asterisk, the convolution sign, rather than the cross of a product sign - three
# strokes through the centre, each again scaled per dimension so the arms stay even.
for angle_deg in (30, 90, 150):
    dx = 0.62 * OP_R_CM * np.cos(np.radians(angle_deg)) / FIG_W_CM
    dy = 0.62 * OP_R_CM * np.sin(np.radians(angle_deg)) / FIG_H_CM
    ax_bg.plot(
        [op_x - dx, op_x + dx], [op_y - dy, op_y + dy], color="k", lw=1.2, zorder=5
    )

# Arrow from operator right to simulated data
ax_bg.annotate(
    "",
    xy=pt_cm(4.64, 11.69),
    xytext=pt_cm(4.07, 11.69),
    arrowprops=dict(
        facecolor="black",
        edgecolor="black",
        width=1.2,
        headwidth=4.5,
        headlength=4.5,
    ),
)

# E. Simulated data trace (Right of schematic)
ax_bg.text(
    *pt_cm(5.76, 12.63),
    "Simulated data",
    fontsize=fontsize_dict["tick"],
    ha="center",
    va="bottom",
)
ax_dff = fig.add_axes(ax_cm(4.77, 11.08, 2.07, 1.32))
ax_dff.plot(time_axis_sphere, sim_dff_sphere, color="blue", linewidth=2)
for trial_id in np.unique(stim_part_sphere[stim_part_sphere > 0]):
    mask = stim_part_sphere == trial_id
    ax_dff.axvspan(
        time_axis_sphere[mask][0],
        time_axis_sphere[mask][-1],
        color="gray",
        alpha=0.1,
    )
for spine in ax_dff.spines.values():
    spine.set_visible(False)
ax_dff.set_xticks([])
ax_dff.set_yticks([])


# =========================================================================
# ROW 1, RIGHT (B): MATRIX GRID, 4 columns x the 2 conditions
# =========================================================================
MAT_W_CM, MAT_GAP_CM, MAT_X0_CM, MAT_H_CM = 1.40, 0.35, 8.4, 2.67
row_y = {
    "free_mat": (12.30 / FIG_H_CM, MAT_H_CM / FIG_H_CM),
    "treadmill_mat": (9.48 / FIG_H_CM, MAT_H_CM / FIG_H_CM),
}

col_w = MAT_W_CM / FIG_W_CM
gap = MAT_GAP_CM / FIG_W_CM
col_x_base = MAT_X0_CM / FIG_W_CM
col_x = {
    "data": col_x_base,
    "real": col_x_base + (col_w + gap),
    "circ": col_x_base + 2 * (col_w + gap),
    "sim": col_x_base + 3 * (col_w + gap),
}
vmin, vmax = 0, 0.8

# Free locomotion row
y0, h = row_y["free_mat"]
ax_mat_real_free = fig.add_axes([col_x["data"], y0, col_w, h])
plot_RS_OF_matrix(
    trials_df=trials_df_sphere,
    roi=roi,
    is_closed_loop=1,
    ax=ax_mat_real_free,
    cbar_width=None,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_mat_real_free.set_ylabel("")
ax_mat_real_free.set_xlabel("")
ax_mat_real_free.set_xticklabels([])
ax_mat_real_free.set_title("Data", fontsize=fontsize_dict["label"])

ax_fit_real_free = fig.add_axes([col_x["real"], y0, col_w, h])
plot_RS_OF_fit(
    neurons_df=ndf,
    roi=roi,
    model="g2d",
    sfx="",
    ax=ax_fit_real_free,
    cbar_width=None,
    label_r2=False,
    vmin=vmin,
    vmax=vmax,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_fit_real_free.set_ylabel("")
ax_fit_real_free.set_yticklabels([])
ax_fit_real_free.set_xlabel("")
ax_fit_real_free.set_xticklabels([])
ax_fit_real_free.set_title("Fit", fontsize=fontsize_dict["label"])

ax_fit_circ_free = fig.add_axes([col_x["circ"], y0, col_w, h])
plot_RS_OF_fit(
    neurons_df=ndf,
    roi=roi,
    model="g2d",
    sfx="_circular_sim_free",
    ax=ax_fit_circ_free,
    cbar_width=None,
    label_r2=False,
    vmin=vmin,
    vmax=vmax,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_fit_circ_free.set_ylabel("")
ax_fit_circ_free.set_yticklabels([])
ax_fit_circ_free.set_xlabel("")
ax_fit_circ_free.set_xticklabels([])
ax_fit_circ_free.set_title("Circularised fit", fontsize=fontsize_dict["label"])

ax_mat_sim_free = fig.add_axes([col_x["sim"], y0, col_w, h])
plot_RS_OF_matrix(
    trials_df=trials_df_sphere_simul,
    roi=roi,
    is_closed_loop=1,
    ax=ax_mat_sim_free,
    cbar_width=None,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_mat_sim_free.set_ylabel("")
ax_mat_sim_free.set_yticklabels([])
ax_mat_sim_free.set_xlabel("")
ax_mat_sim_free.set_xticklabels([])
ax_mat_sim_free.set_title("Simulated data", fontsize=fontsize_dict["label"])

for mat_ax in (ax_mat_real_free, ax_mat_sim_free):
    mat_ax.images[0].set_clim(0, 0.8)
add_matrix_colorbar(fig, ax_mat_sim_free, 0, 0.8, fontsize_dict)

# Motorized wheel row
y0, h = row_y["treadmill_mat"]
ax_mat_real = fig.add_axes([col_x["data"], y0, col_w, h])
plot_RS_OF_matrix(
    trials_df=trials_df_tm,
    roi=roi,
    is_closed_loop=1,
    ax=ax_mat_real,
    cbar_width=None,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_mat_real.set_xlabel("")

ax_fit_real = fig.add_axes([col_x["real"], y0, col_w, h])
plot_RS_OF_fit(
    neurons_df=ndf,
    roi=roi,
    model="g2d",
    sfx="_treadmill",
    ax=ax_fit_real,
    cbar_width=None,
    label_r2=False,
    vmin=vmin,
    vmax=vmax,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_fit_real.set_ylabel("")
ax_fit_real.set_yticklabels([])
ax_fit_real.set_xlabel("")

ax_fit_circ = fig.add_axes([col_x["circ"], y0, col_w, h])
plot_RS_OF_fit(
    neurons_df=ndf,
    roi=roi,
    model="g2d",
    sfx="_circular_sim",
    ax=ax_fit_circ,
    cbar_width=None,
    label_r2=False,
    vmin=vmin,
    vmax=vmax,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_fit_circ.set_ylabel("")
ax_fit_circ.set_yticklabels([])
ax_fit_circ.set_xlabel("")

ax_mat_sim = fig.add_axes([col_x["sim"], y0, col_w, h])
plot_RS_OF_matrix(
    trials_df=trials_df_tm_simul,
    roi=roi,
    is_closed_loop=1,
    ax=ax_mat_sim,
    cbar_width=None,
    fontsize_dict=fontsize_dict,
    **range_kwargs,
)
ax_mat_sim.set_ylabel("")

# The y label was the lower-left matrix's own, so it sat centred on that row alone. Drawn
# on `ax_bg` instead, centred on the two rows together.
ax_mat_real.set_ylabel("")
ax_bg.text(
    *pt_cm(7.75, (9.48 + 12.30 + MAT_H_CM) / 2),
    "Optic flow speed (°/s)",
    fontsize=fontsize_dict["label"],
    rotation=90,
    ha="center",
    va="center",
)
ax_mat_sim.set_yticklabels([])
ax_mat_sim.set_xlabel("")

for mat_ax in (ax_mat_real, ax_mat_sim):
    mat_ax.images[0].set_clim(0, 0.8)
add_matrix_colorbar(fig, ax_mat_sim, 0, 0.8, fontsize_dict)

ax_bg.text(
    *pt_cm(7.15, 9.48 + MAT_H_CM / 2),
    "Motorized wheel",
    rotation=90,
    ha="center",
    va="center",
    fontsize=fontsize_dict["label"],
)
ax_bg.text(
    *pt_cm(7.15, 12.30 + MAT_H_CM / 2),
    "Free locomotion",
    rotation=90,
    ha="center",
    va="center",
    fontsize=fontsize_dict["label"],
)
ax_bg.text(
    *pt_cm(MAT_X0_CM + 2 * MAT_W_CM + 1.5 * MAT_GAP_CM, 8.75),
    "Running speed (cm/s)",
    ha="center",
    va="center",
    fontsize=fontsize_dict["label"],
)


# =========================================================================
# ROW 2 (C): POPULATION POLAR PLOTS, one under each panel above
# =========================================================================
POLAR_ELONGATION_CUTOFF = 1.4
POLAR_IN_COLOR, POLAR_OUT_COLOR = "k", "darkred"
sc_kwargs = dict(s=15, alpha=0.7, linewidths=0, clip_on=False, zorder=3)

# Centred under the schematic (spans 0.45-6.84 cm) and under the matrix grid
# (9.2-17.3 cm) respectively, and twice the width they had beside the von Mises panels.
POLAR_W_CM, POLAR_H_CM, POLAR_Y0_CM = 6.0, 5.8, 0.7

ax_free = fig.add_axes(
    ax_cm(0.65, POLAR_Y0_CM, POLAR_W_CM, POLAR_H_CM), projection="polar"
)
ax_motor = fig.add_axes(
    ax_cm(9.40, POLAR_Y0_CM, POLAR_W_CM, POLAR_H_CM), projection="polar"
)

# The motorised-wheel panel uses the simulated fits run with the SAME configuration as
# the real trial-average plateau fits the depth-cells figure reads
# (`precompute_data/fit_revision_simulation.py`): trial averages, plateau onsets,
# TREADMILL_PARAM_RANGE, niter=10, max_rs2motor_diff=0.3, seeded from the trial-average
# plateau popts. That matters for this control: under the old per-frame configuration a
# *circular* ground truth came back with 33% of fits above this cut, matched it is 13%, so
# the per-frame family overstated how much elongation the calcium dynamics and the stimulus
# sampling geometry alone produce.
#
# Free locomotion has no matched counterpart - the new fits are treadmill-only - so that
# panel keeps the per-frame family. The per-frame treadmill version is in the cell below.
TA_SIM = treadmill.TA_SIM_SUFFIX  # "_trial_average_plateau"

for pax, simul_df, sfx, popt_col, theta_col, title in zip(
    [ax_free, ax_motor],
    [simul_df_spheres, simul_df_treadmill],
    ["", "_treadmill"],
    ["popt_simulated", f"popt_simulated{TA_SIM}"],
    ["g2d_theta", f"g2d_theta{TA_SIM}"],
    ["Free locomotion", "Motorized wheel"],
):
    # Population selection is unchanged in both panels: the *real* per-frame R^2 flag.
    # Matching that to the trial-average family too would need
    # `add_trial_average_rsof_columns` here, as figure_depth_cells.ipynb does.
    ndf_pop = neurons_df[neurons_df[f"rsof_neuron{sfx}"]]
    ndf_sim = simul_df[simul_df.roi_uid.isin(ndf_pop.roi_uid)].dropna(
        subset=[theta_col]
    )
    elongation = ndf_sim[popt_col].apply(fit_gb.get_semimajor_length) / ndf_sim[
        popt_col
    ].apply(fit_gb.get_semiminor_length)
    elong_ok = elongation > POLAR_ELONGATION_CUTOFF

    plot_angle_eccentricity_polar(
        pax,
        ndf_sim.loc[elong_ok, theta_col].astype(float),
        None,
        fontsize_dict,
        radial_scale="elongation",
        axis_ratio=elongation[elong_ok],
        scale=0.60,
        rasterize_schematics="each",
        c=POLAR_IN_COLOR,
        **sc_kwargs,
    )
    pax.scatter(
        np.radians(ndf_sim.loc[~elong_ok, theta_col].astype(float)),
        np.clip(np.log2(elongation[~elong_ok]), 0, None),
        c=POLAR_OUT_COLOR,
        **sc_kwargs,
    )
    print(
        f"{title}: {int(elong_ok.sum())}/{len(ndf_sim)} simulated fits with "
        f"elongation > {POLAR_ELONGATION_CUTOFF} "
        f"(median {elongation.median():.2f}), from {popt_col}"
    )

# Condition labels placed by hand rather than with `set_title`: the shape insets that
# `plot_angle_eccentricity_polar` draws sit ~1 cm outside the axes, so an axes-anchored
# title lands on the 90 deg inset at any pad. Centred on each polar axes.
for label, x_cm in (("Free locomotion", 3.65), ("Motorized wheel", 12.40)):
    ax_bg.text(
        *pt_cm(x_cm, 7.75),
        label,
        fontsize=fontsize_dict["title"],
        ha="center",
        va="bottom",
    )

# Panel letters, at the top-left of each row (same convention as figure_depth_cells).
for letter, x_cm, y_cm in (
    ("A", 0.15, 15.45),
    ("B", 7.8, 15.45),
    ("C", 0.15, 7.9),
    ("D", 8.90, 7.9),
):
    fig.text(
        *pt_cm(x_cm, y_cm),
        letter,
        fontsize=fontsize_dict.get("panel", 10),
        fontweight="bold",
        ha="left",
        va="top",
    )

style.savefig(
    SAVE_ROOT / "fig_supp_simulation_control.svg",
    bbox_inches="tight",
    transparent=True,
    fig=fig,
)
print(f"Saved figure to {SAVE_ROOT / 'fig_supp_simulation_control.svg'}")

# Supplementary analysis


In [ ]:
# REFERENCE: the PER-FRAME version of the motorised-wheel panels above.
#
# The figure now uses the simulated fits run with the real trial-average plateau config
# (`precompute_data/fit_revision_simulation.py`). This cell keeps the family that
# `treadmill.simulate_and_fit_session` produced - per frame, wide DEFAULT_PARAM_RANGE,
# niter=5, max_rs2motor_diff=0.5, k_folds=1, seeded from the per-frame
# `rsof_popt_closedloop_g2d_treadmill` popts - which is what the figure showed until now,
# so the two can be compared and the older numbers reproduced.
#
# The two differ substantially, and in a direction that matters: the ground truth here is
# *circular* by construction, so every fit above the elongation cut is an artefact of the
# calcium dynamics plus the stimulus sampling geometry. Pooled over the four sessions the
# per-frame fits put 33% of a circular population above 1.4:1 (median elongation 1.23),
# the matched fits 13% (median 1.16) - i.e. this per-frame control overstates the artefact
# by about 2.5x, because the trial-average fit with the narrowed bounds is better
# constrained.
#
# Otherwise unchanged: same selection logic (significant treadmill RS/OF fit, elongated
# ellipses only), same axial von Mises mixture with k chosen by BIC, same bins as the
# depth-cells figure, so the mixtures remain directly comparable to the real data.
from v1_depth_map.figure_utils import von_mises as vm

SIM_ELONGATION_CUTOFF = 1.4  # same cut as the real-data panel: below this elongation
# (sigma_major / sigma_minor) the ellipse has no meaningful orientation
MAX_K = 5
N_ANGLE_BINS = 37
angle_edges = np.linspace(-45, 135, N_ANGLE_BINS + 1)
angle_bin_w = np.diff(angle_edges)[0]
angle_grid = np.linspace(-45, 135, 400)
VONMISES_COLOR = "#000000"
COMPONENT_COLORS = ("#E69F00", "#56B4E9", "#7F3C8D", "#009E73", "#0072B2")


def wrap_axial_deg(deg):
    """Wrap an axial angle to [-45, 135), matching fit_gb.get_gaussian_angle."""
    return (np.asarray(deg) + 45) % 180 - 45


# Same population as the motorised-wheel polar scatter above.
ndf_pop_tm = neurons_df[neurons_df["rsof_neuron_treadmill"].fillna(False)]
sim_tm = simul_df_treadmill[simul_df_treadmill.roi_uid.isin(ndf_pop_tm.roi_uid)].dropna(
    subset=["g2d_theta_treadmill", "popt_simulated"]
)
# The simulated dataframe carries the fit parameters rather than the axis lengths, so
# derive the elongation the way the polar panel above does.
sim_elongation = sim_tm.popt_simulated.apply(
    fit_gb.get_semimajor_length
) / sim_tm.popt_simulated.apply(fit_gb.get_semiminor_length)
sim_elong_ok = sim_elongation > SIM_ELONGATION_CUTOFF
angles_deg = wrap_axial_deg(
    sim_tm.loc[sim_elong_ok, "g2d_theta_treadmill"].astype(float).to_numpy()
)

vm_results = vm.model_selection(
    np.radians(angles_deg), max_k=MAX_K, seed=42, verbose=False
)
k_vm = min(vm_results, key=lambda k: vm_results[k]["bic"])
r_vm = vm_results[k_vm]

fig_vm, (ax_bic, ax_hist) = plt.subplots(1, 2, figsize=(13 * cm, 6 * cm))

# Left: how many components the distribution needs. Lower BIC is better; the dotted
# line marks the k drawn on the right.
ks = sorted(vm_results)
ax_bic.plot(ks, [vm_results[k]["bic"] for k in ks], "o-", color="#CC79A7", ms=3, lw=1)
ax_bic.axvline(k_vm, color="k", ls=":", lw=1)
ax_bic.set_xticks(ks)
ax_bic.set_xlabel("n components", fontsize=fontsize_dict["label"])
ax_bic.set_ylabel("BIC", fontsize=fontsize_dict["label"])
ax_bic.set_title(f"Best k = {k_vm}", fontsize=fontsize_dict["title"])
ax_bic.tick_params(axis="both", labelsize=fontsize_dict["tick"], pad=1)
ax_bic.spines[["top", "right"]].set_visible(False)

# Right: the orientation histogram with the best-BIC mixture on top.
ax_hist.hist(
    angles_deg, bins=angle_edges, color="lightgrey", edgecolor="k", linewidth=0.5
)

# Density -> counts, so the mixture curve can be overlaid on the raw histogram.
to_counts = len(angles_deg) * angle_bin_w

pi_k, mu_k, kappa_k = (
    np.asarray(r_vm["pi_k"], dtype=float),
    np.asarray(r_vm["mu_k"], dtype=float),
    np.asarray(r_vm["kappa_k"], dtype=float),
)


def component_counts(idx):
    """Counts curve of the components in `idx`, i.e. their weighted density."""
    return (
        vm.axial_mixture_density(
            np.radians(angle_grid), pi_k[idx], mu_k[idx], kappa_k[idx]
        )
        / np.degrees(1)
        * to_counts
    )


ax_hist.plot(
    angle_grid,
    component_counts(slice(None)),
    color=VONMISES_COLOR,
    lw=1.2,
    zorder=3,
    label=f"von Mises mixture (k={k_vm})",
)

mu_deg_vm = wrap_axial_deg(np.degrees(mu_k))
for rank, i_comp in enumerate(np.argsort(mu_deg_vm)):
    ax_hist.plot(
        angle_grid,
        component_counts([i_comp]),
        color=COMPONENT_COLORS[rank % len(COMPONENT_COLORS)],
        lw=0.8,
        ls="--",
        zorder=2,
        label=f"{mu_deg_vm[i_comp]:.0f}°: {100 * pi_k[i_comp]:.0f}%",
    )

ax_hist.set_xlim(-45, 135)
ax_hist.set_xticks([-45, 0, 45, 90, 135])
ax_hist.set_xlabel("Ellipse orientation (deg)", fontsize=fontsize_dict["label"])
ax_hist.set_ylabel("Number of cells", fontsize=fontsize_dict["label"])
ax_hist.tick_params(axis="both", labelsize=fontsize_dict["tick"], pad=1)
ax_hist.set_title(
    "Simulated, motorized wheel (per-frame fit)",
    fontsize=fontsize_dict["title"],
)
ax_hist.legend(fontsize=fontsize_dict["legend"], frameon=False, loc="upper left")
ax_hist.spines[["top", "right"]].set_visible(False)
fig_vm.tight_layout()

print(
    f"PER-FRAME reference: elongation median {sim_elongation.median():.2f}, "
    f"{(sim_elongation > SIM_ELONGATION_CUTOFF).mean() * 100:.0f}% above "
    f"{SIM_ELONGATION_CUTOFF} (matched fits, for comparison: 1.16 and 13%)"
)
print(
    f"orientation mixture on {len(angles_deg)}/{len(sim_tm)} simulated neurons "
    f"(elongation > {SIM_ELONGATION_CUTOFF}): von Mises k={k_vm}"
)
for k in ks:
    print(
        f"  k={k}  BIC={vm_results[k]['bic']:10.1f}"
        + ("   <-- best" if k == k_vm else "")
    )
for i_comp in np.argsort(mu_deg_vm):
    print(
        f"  mu={mu_deg_vm[i_comp]:6.1f} deg  kappa={kappa_k[i_comp]:6.2f}"
        f"  {100 * pi_k[i_comp]:5.1f}% of the population"
    )

In [ ]:
# Plot an example ellipse schematic with the elongation corresponding to the cutoff used
# above, so the reader can see what "elongation > 1.4" means in practice. The cut is on
# sigma_major / sigma_minor, so only the shape matters here: every panel holds the same
# minor axis and stretches the major one, which is what the elongation axis of the polar
# panels measures ("a circle pulled along one direction"). All are drawn at a tuning
# angle of 45 deg, the depth (RS/OF ratio) axis, as the polar schematics are.
from matplotlib.patches import Ellipse

SCHEMATIC_ELONGATIONS = (1.0, 1.25, SIM_ELONGATION_CUTOFF, 2.0, 3.0)
SCHEMATIC_ANGLE = 45.0  # deg, the depth (RS/OF ratio) axis
SIGMA_MINOR = 1.0  # arbitrary units, only the ratio is meaningful
CUT_COLOR, REF_COLOR = "darkred", "0.5"

# Same limits in every panel, so the growth of the major axis is the visible difference.
half_lim = SIGMA_MINOR * max(SCHEMATIC_ELONGATIONS) * 1.15
fig_el, axes_el = plt.subplots(
    1,
    len(SCHEMATIC_ELONGATIONS),
    figsize=(3.4 * len(SCHEMATIC_ELONGATIONS) * cm, 4.6 * cm),
)

print(f"{'elongation':>10} {'eccentricity':>13} {'flattening':>11} {'log2(a/b)':>10}")
for ax_el, elongation in zip(np.atleast_1d(axes_el), SCHEMATIC_ELONGATIONS):
    at_cut = np.isclose(elongation, SIM_ELONGATION_CUTOFF)
    color = CUT_COLOR if at_cut else REF_COLOR
    # `Ellipse` puts the major axis (height) along +y at angle=0, so `angle = theta - 90`
    # points it `theta` deg up from horizontal, the convention the polar insets use.
    ax_el.add_patch(
        Ellipse(
            (0, 0),
            width=2 * SIGMA_MINOR,
            height=2 * SIGMA_MINOR * elongation,
            angle=SCHEMATIC_ANGLE - 90,
            facecolor=color,
            alpha=0.35,
            edgecolor=color,
            linewidth=0.8,
        )
    )
    # The 45 deg axis the ellipses are drawn along, for reference.
    ax_el.plot(
        [-half_lim, half_lim],
        [-half_lim, half_lim],
        ls=":",
        lw=0.5,
        color="k",
        zorder=0,
    )
    ax_el.set_xlim(-half_lim, half_lim)
    ax_el.set_ylim(-half_lim, half_lim)
    ax_el.set_aspect("equal")
    ax_el.set_xticks([])
    ax_el.set_yticks([])
    for spine in ax_el.spines.values():
        spine.set_linewidth(0.5)
    eccentricity = np.sqrt(1 - 1 / elongation**2)
    ax_el.set_title(
        f"{elongation:g}:1{' (cutoff)' if at_cut else ''}\ne = {eccentricity:.2f}",
        fontsize=fontsize_dict["title"],
        color=color if at_cut else "k",
    )
    print(
        f"{elongation:10.2f} {eccentricity:13.2f} {1 - 1 / elongation:11.2f}"
        f" {np.log2(elongation):10.2f}"
    )

fig_el.suptitle(
    f"Tuning-ellipse shape at and around the {SIM_ELONGATION_CUTOFF}:1 elongation cut"
    f" (drawn at {SCHEMATIC_ANGLE:g}°)",
    fontsize=fontsize_dict["title"],
)
fig_el.tight_layout()

## Figure legend

Science-style legend for this figure, with `n`s and summary statistics filled in from
the data loaded above (edit `FIG_NUM` to the final supplementary figure number).


In [ ]:
# Figure legend, Science style. Numbers are pulled from the loaded data rather than
# typed in, so the legend follows whatever the current fits/selection give.
import textwrap

FIG_NUM = "X"  # supplementary figure number in the manuscript

n_sessions = neurons_df.session.nunique()
n_mice = neurons_df.session.str.split("_").str[0].nunique()

# Same selection and elongation measure as the polar panels (C, D) above.
legend_stats = {}
for cond, simul_df, sfx, popt_col, theta_col in (
    ("free", simul_df_spheres, "", "popt_simulated", "g2d_theta"),
    (
        "motor",
        simul_df_treadmill,
        "_treadmill",
        f"popt_simulated{TA_SIM}",
        f"g2d_theta{TA_SIM}",
    ),
):
    ndf_pop = neurons_df[neurons_df[f"rsof_neuron{sfx}"]]
    ndf_sim = simul_df[simul_df.roi_uid.isin(ndf_pop.roi_uid)].dropna(
        subset=[theta_col]
    )
    elong = ndf_sim[popt_col].apply(fit_gb.get_semimajor_length) / ndf_sim[
        popt_col
    ].apply(fit_gb.get_semiminor_length)
    ok = elong > POLAR_ELONGATION_CUTOFF
    legend_stats[cond] = dict(
        n=len(ndf_sim),
        n_ok=int(ok.sum()),
        pct=100 * ok.mean(),
        median=elong.median(),
    )

free, motor = legend_stats["free"], legend_stats["motor"]


# LaTeX bits kept as separate variables so the f-strings below stay free of the
# doubled braces an inline `\mathrm{...}` would need.
def bf(s):
    """Bold, for the panel letters and the title sentence."""
    return r"\textbf{" + s + "}"


TAU_RISE = r"\tau_{\mathrm{rise}}"
TAU_DECAY = r"\tau_{\mathrm{decay}}"
SIGMA_RATIO = r"\sigma_{\mathrm{major}}/\sigma_{\mathrm{minor}}"
DFF = r"\Delta F/F"

legend = [
    bf(f"Fig. S{FIG_NUM}.")
    + " "
    + bf(
        "Motorized wheel controls for the contribution of calcium indicator dynamics"
        " to RS/OF tuning-ellipse elongation."
    ),
    f"{bf('(A)')} Simulation procedure. The measured RS/OF Gaussian fit of a neuron "
    "is circularised (major axis reduced to the minor axis), evaluated along the "
    "running speed and optic flow trajectory actually experienced during the "
    "recording, and convolved with a biexponential calcium kernel "
    "to give a simulated "
    f"${DFF}$ trace, which is then refitted exactly like the real data.",
    f"{bf('(B)')} Example neuron during free locomotion (top) and on the motorized "
    "wheel (bottom)",
    f"{bf('(C')} and {bf('D)')} Orientation and elongation (${SIGMA_RATIO}$) of the "
    "ellipses fitted to the simulated responses of all RS/OF-tuned neurons during "
    f"free locomotion (C; $n = {free['n']}$ neurons) and on the motorized wheel "
    f"(D; $n = {motor['n']}$ neurons), from ${n_sessions}$ sessions in ${n_mice}$ "
    f"mice. Black, ellipses with elongation $> {POLAR_ELONGATION_CUTOFF}$ "
    rf"(${free['n_ok']}/{free['n']}$, {free['pct']:.0f}\% in C; "
    rf"${motor['n_ok']}/{motor['n']}$, {motor['pct']:.0f}\% in D); dark red, "
    "ellipses below that cut-off.",
]

# LaTeX source, ready to paste: one paragraph per blank-line-separated block.
print("\n\n".join(textwrap.fill(p, 88) for p in legend))